# Droplet rebound simulation (gauthier experiments) - Run

This notebook is used to run the actual simulations for the droplet rebound test case. The simulation will be restarted from a previously computed initial state, see `DropletReboundInit.ipynb`

In [1]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [44]:
using System.IO;
using MPI.Wrappers;
using NUnit.Framework;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;

In [3]:
Console.WriteLine("Execution Date/time is " + DateTime.Now);

Execution Date/time is 5/8/2025 10:58:02 AM


In [4]:
string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

In [5]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client LocalPC @D:\local\binaries
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries"
2,BoSSS.Application.BoSSSpad.SlurmClient,"SlurmClient Lb2-specialPrj : nu39gihu@lcluster17.hrz.tu-darmstadt.de, Slurm account: special00006"


In [6]:
var myBatch = ExecutionQueues[1];

if(userName.Equals(@"FDY\JenkinsCI"))
    myBatch = GetDefaultQueue();

myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfAdditionalServiceCores,0
NumOfAdditionalServiceCoresMPISerial,0
NumOfServiceCoresPerMPIprocess,0
ServerName,DC3


In [7]:
BoSSSshell.WorkflowMgm.Init("DropletRebound_Gauthier", myBatch);

Project name is set to 'DropletRebound_Gauthier'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\DropletRebound_Gauthier'.


In [8]:
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

## Setting up restart

Choose if this notebook should do a restart. 

In [9]:
bool restartRun = true;

In [10]:
OpenOrCreateDatabase(@"\\dc3\userspace\smuda\hpccluster\DropletRebound_Gauthier");
//OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\DropletRebound_Gauthier");
//OpenOrCreateDatabase(@"D:\local\DropletRebound_Gauthier");

In [ ]:
var sessions = databases.Pick(0).Sessions;
sessions

#0: DropletRebound_Gauthier	DropletReboundGauthier_2D_8x8AMR1_k3_DongOutflow_useBCMap_ReInit10_test3*	3/12/2025 2:46:25 PM	2b101a2a...
#1: DropletRebound_Gauthier	DropletReboundGauthier_2D_8x8AMR1_k3_DongOutflow_useBCMap_ReInit10_test2*	3/12/2025 2:37:47 PM	9741cd06...
#2: DropletRebound_Gauthier	DropletReboundGauthier_2D_8x8AMR1_k3_DongOutflow_useBCMap_ReInit10*	3/6/2025 2:44:57 PM	d92bdd60...
#3: DropletRebound_Gauthier	DropletReboundGauthier_2D_8x8AMR1_k3_DongOutflow_useBCMap_ReInit10*	3/6/2025 1:54:20 PM	c6a7f987...
#4: DropletRebound_Gauthier	DropletReboundGauthier_2D_48x16AMR1_k3_velocityInletTop_useBCMap_ReInit10*	3/6/2025 9:52:56 AM	34254bb4...
#5: DropletRebound_Gauthier	DropletReboundGauthier_2D_24x16AMR1_k3_velocityInletTop_useBCMap_ReInit10*	3/5/2025 2:18:30 PM	21a4ef85...
#6: DropletRebound_Gauthier	DropletReboundGauthier_2D_48x32AMR2_k3_useBCMap_ReInit10_Picard_restart1*	2/20/2025 10:25:48 AM	ad20f851...
#7: DropletRebound_Gauthier	DropletReboundGauthier_2D_48x32AMR2_k3_u

In [13]:
var selection = sessions.Where(sess => sess.Name.Contains("DropletReboundGauthier_8x8x8AMR0_k3_ReI4_restart1"));
selection

#0: DropletRebound_Gauthier	DropletReboundGauthier_8x8x8AMR0_k3_ReI4_restart1*	7/1/2024 9:08:59 AM	57e8f3ff...


In [ ]:
//BoSSSshell.WorkflowMgm.Sessions
//sessions.Pick(32).RestartedFrom

In [ ]:
//BoSSSshell.WorkflowMgm.Sessions.Pick(0).Delete(true)
//selection.Pick(0).Copy(databases.Pick(4))

In [ ]:
// // check for backup on userspace database
// var backupDB = OpenOrCreateDatabase(@"\\dc3\userspace\smuda\hpccluster\DropletRebound_Gauthier");
// List<ISessionInfo> noBackUpSessions = new List<ISessionInfo>();
// foreach(var sess in sessions) {
//     if(!backupDB.Sessions.Contains(sess)) {
//         noBackUpSessions.Add(sess);
//     }
// }
// noBackUpSessions

In [ ]:
//noBackUpSessions.MoveAll(backupDB);

In [14]:
ISessionInfo restartSession;
if(restartRun == true) {
    //restartSession = sessions.Where(sess => sess.Name.Contains("DropletReboundGauthier_8x8x8AMR1_k3_ReI4_restart4")).Single();
    //restartSession = sessions.Where(sess => sess.Name.Contains("DropletReboundGauthier_8x12x8AMR1_k2_ReI4_restart2")).Single();
    restartSession = selection.Pick(0);
    //restartSession = databases.Pick(0).Sessions.Pick(4);
}
restartSession

DropletRebound_Gauthier	DropletReboundGauthier_8x8x8AMR0_k3_ReI4_restart1*	7/1/2024 9:08:59 AM	57e8f3ff...

In [ ]:
// List<ISessionInfo> restartSessionList = new List<ISessionInfo>();
// restartSessionList.Add(restartSession);
// Guid restartID = restartSession.RestartedFrom;
// while(restartID != Guid.Empty) {
//     var rSess = sessions.Where(sess => sess.ID == restartID).Single();
//     restartSessionList.Add(rSess);
//     restartID = rSess.RestartedFrom;
// }
// restartSessionList

In [ ]:
//restartSessionList.CopyAll(databases.Pick(1));
//restartSessionList.Pick(2).Timesteps.Skip(0).Export().WithSupersampling(2).Do();

In [ ]:
//restartSession.GetSessionDirectory()
//restartSession.Copy(databases.Pick(1));

In [ ]:
//restartSession.Timesteps.Count

In [ ]:
//restartSession.Timesteps.Skip(0).Export().WithSupersampling(2).Do()

automated naming of restart sessions

In [15]:
string restartName = (restartSession != null) ? String.Empty : "noRestart";

In [16]:
if(restartSession != null) {
Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
procSIs.Push(restartSession);
var currSI = restartSession;
var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
while(!rSIs.IsNullOrEmpty()) {
    var rSI = rSIs.Single();
    procSIs.Push(rSI);
    currSI = rSI;
    rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
}
int restartNum = procSIs.Count;

string orgName = restartSession.Name;
if (restartNum == 1) {
    //restartName = orgName.Substring(0, orgName.Length - 10); // remoinvg _InitState
//} else if (restartNum == 1){
    restartName = orgName.Substring(0, orgName.Length) + "_restart1"; // + restartNum; 
} else {
    restartName = orgName.Substring(0, orgName.Length - 1) + restartNum; 
}
}
restartName

DropletReboundGauthier_8x8x8AMR0_k3_ReI4_restart2

In [17]:
restartName = "DropletReboundGauthier_8x8x8AMR0_k3_ReI4_restart2_DongBC";

## Boundary Conditions

0: pressure outlet, 1: velocity inlet

In [18]:
static int BC_tag2_top = 0;
static int BC_tag4_front = 0;

## Grid creation

The used grid is a cartesain box around the droplet injection location, which is located at `radiusOP` (in $r$-direction) away from the center of the rotating disk.

In [19]:
static public Grid3D RotatingDiskSector_CartesianCutOut(double radiusOP, double l_radial, double l_upstream, double l_downstream, double h_axial, int res_radial, int res_azimuthal, int res_axial) {

    double[] xNodes = GenericBlas.Linspace(radiusOP - (l_radial / 2.0), radiusOP + (l_radial / 2.0), res_radial + 1);    // radial direction
    double[] yNodes = GenericBlas.Linspace(-l_upstream, l_downstream, res_azimuthal + 1);    // azimuthal direction
    // double[] yNodes = GenericBlas.Linspace(-l_azimuthal / 3.0, (2.0 * l_azimuthal) / 3.0, res_azimuthal + 1);    // azimuthal direction
    double[] zNodes = GenericBlas.Linspace(0.0, h_axial, res_axial + 1);    // axial direction
    
    var grd = Grid3D.Cartesian3DGrid(xNodes, yNodes, zNodes, periodicY: false);
    grd.Name = $"RotatingDiskSector3D_CartesianCutOut_{res_radial}x{res_azimuthal}x{res_axial}";

    grd.EdgeTagNames.Add(1, "velocity_inlet_rotatingDisk");

    if (BC_tag2_top == 0)
        grd.EdgeTagNames.Add(2, "pressure_outlet_top");
    if (BC_tag2_top == 1)
        grd.EdgeTagNames.Add(2, "velocity_inlet_top");

    grd.EdgeTagNames.Add(3, "velocity_inlet_back");

    if (BC_tag4_front == 0)
        grd.EdgeTagNames.Add(4, "pressure_outlet_front");
    if (BC_tag4_front == 1)
        grd.EdgeTagNames.Add(4, "velocity_inlet_front");
   
    grd.EdgeTagNames.Add(5, "velocity_inlet_upstream");
    grd.EdgeTagNames.Add(6, "pressure_outlet_downstream");

    grd.DefineEdgeTags(delegate (Vector X) {
        byte et = 0;
        if (X.z.Abs() <= 1e-8)
            et = 1;
        if ((X.z - h_axial).Abs() <= 1e-8)
            et = 2;
        if (((X.x - radiusOP) + (l_radial / 2.0)).Abs() <= 1e-8)
            et = 3;
        if (((X.x - radiusOP) - (l_radial / 2.0)).Abs() <= 1e-8)
            et = 4;
        if ((X.y + (l_upstream)).Abs() <= 1e-8)
            et = 5;
        if ((X.y - (l_downstream)).Abs() <= 1e-8)
            et = 6;
        // if ((X.y + (l_azimuthal / 3.0)).Abs() <= 1e-8)
        //     et = 5;
        // if ((X.y - (2.0 * l_azimuthal / 3.0)).Abs() <= 1e-8)
        //     et = 6;

        return et;
    });    

    return grd;
}

## Simulation setup

In [20]:
double radiusOP = 90e-3; // operating point (droplet injection point) -> Re = radiusOp / Lstar
double density = 1.2;  
double kinViscosity = 2.0e-5 / density; // kinematic viscosity
//double omega = 46.11514476; // rotation rate
double velOP = 38.0; 
double omega = velOP / radiusOP; //46.11514476; // rotation rate
double Lstar = Math.Sqrt(kinViscosity / omega);  // used for non-dimensionalization of the flow fields
Lstar

0.00019867985355975657

Reynolds number at the point of injection

In [21]:
double reynolds = radiusOP / Lstar;
reynolds

452.990066116245

Boundary layer (BL) thickness

In [22]:
double zBL = 5.5;
double zBLstar = zBL * Lstar;
zBLstar

0.001092739194578661

In [23]:
double radiusDrop = 0.91e-3;
double topDropPos = zBLstar + (2.0*radiusDrop);
topDropPos

0.002912739194578661

Height of the computational domain 

In [24]:
double zTop = 20;
double zTopStar = zTop * Lstar;
zTopStar

0.003973597071195131

## Prescribed boundary conditions for the flow field 

The B.C. are given by the Homotopy Analysis method (HAM), which give a linear combination in term of $\sum_{n,i} \textrm{e}^{-n \eta} \eta^{i}$, where $\eta$ describes the dimensionless height in axial-direction.

In [25]:
using BoSSS.Application.XNSE_Solver.SpecificSolutions;

In [26]:
MultidimensionalArray HAMcoeff_velU = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffU.txt");
MultidimensionalArray HAMcoeff_velV = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffV.txt");
MultidimensionalArray HAMcoeff_velW = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffW.txt");
MultidimensionalArray HAMcoeff_P = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffP.txt");

In [28]:
var vonKarmanHAM_velX = new RotatingDiskBoundaryLayerConditionsHAM_VelocityX();
vonKarmanHAM_velX.SetData(HAMcoeff_velU, HAMcoeff_velV, omega, kinViscosity);
var vonKarmanHAM_velY = new RotatingDiskBoundaryLayerConditionsHAM_VelocityY();
vonKarmanHAM_velY.SetData(HAMcoeff_velU, HAMcoeff_velV, omega, kinViscosity);
var vonKarmanHAM_velZ = new RotatingDiskBoundaryLayerConditionsHAM_VelocityZ();
vonKarmanHAM_velZ.SetData(HAMcoeff_velW, omega, kinViscosity);
var vonKarmanHAM_P = new RotatingDiskBoundaryLayerConditionsHAM_PressureP();
vonKarmanHAM_P.SetData(HAMcoeff_P, omega, kinViscosity, density);

### B.C. for the rotating disk

One may also use the B.C. for the analytic solution

In [29]:
omega

422.22222222222223

In [30]:
Formula RotatingDiskVelocityX = new Formula(
    "VelX",
    false,
    "double VelX(double[] X) { " + 
    "double omega = 422.22222222222223; "  + 
    "double theta = Math.Atan2(X[1], X[0]); "  + 
    "double rOnRotDisk = Math.Sqrt(X[0].Pow2() + X[1].Pow2()); " + 
    "double velX = - rOnRotDisk * Math.Sin(theta) * omega;" +
    "return velX; } "
);

In [31]:
Formula RotatingDiskVelocityY = new Formula(
    "VelY",
    false,
    "double VelY(double[] X) { " + 
    "double omega = 422.22222222222223; "  + 
    "double theta = Math.Atan2(X[1], X[0]); "  + 
    "double rOnRotDisk = Math.Sqrt(X[0].Pow2() + X[1].Pow2()); " + 
    "double velY = rOnRotDisk * Math.Cos(theta) * omega;" +
    "return velY; } "
);

### B.C. for the top boundary 

Again, one may use the B.C. for the analytic solution

In [32]:
double velW_top = -0.88447342;
double velWstar_top = velW_top * Math.Sqrt(kinViscosity * omega); 
velWstar_top

-0.07419586537108543

In [33]:
Formula VelocityZ_top = new Formula(
    "VelZ",
    false,
    "double VelZ(double[] X) { return -0.07419586537108543; } "
);

## Initial conditions for the injected droplet 

If not restarted from precomputed initial state with impact velocity for the droplet.

In [34]:
radiusOP

0.09

In [35]:
radiusDrop

0.00091

In [36]:
double initHeight = zBLstar + radiusDrop;
initHeight

0.0020027391945786613

In [37]:
Formula PhiFunc = new Formula(
    "Phi",
    false,
    "double Phi(double[] X) { " +
    "double radiusOP = 0.09;" +
    "double radiusDrop = 0.91e-3;" +   
    "double initHeight = 1.5e-3;" + 
    "return Math.Sqrt((X[0] - radiusOP).Pow2() + X[1].Pow2() + (X[2] - initHeight).Pow2()) - radiusDrop; } "
);

In [38]:
Formula InitVelocity = new Formula(
    "VelZ",
    false,
    "double VelZ(double[] X) { return -0.23; } "
);

In [39]:
Formula GravityValue = new Formula(
    "GravZ",
    false,
    "double GravZ(double[] X) { return -9.81; } "
);

## Control settings

In [41]:
double densityDrop = 960;
double sigma = 21e-3;

int res_global = 8;
int AMRlevel_disk = 0;
int AMRlevel_drop = 0;
int AMRlevel_dropBL = 0;
int maxAMRlevel = (new int[] {AMRlevel_disk, AMRlevel_drop, AMRlevel_dropBL}).Max();
double hmin = (zTopStar / res_global) / (maxAMRlevel + 1);

int k = 3;

double dt_sigma = BoSSS.Solution.XNSECommon.XNSEUtils.GetCapillaryTimeStep(densityDrop, density, sigma, hmin, k+1);
dt_sigma

8.450707167249377E-05

In [43]:
List<XNSE_Control> Controls = new List<XNSE_Control>();
Controls.Clear();

In [51]:
if (restartRun) {
    
    string sessionDir = restartSession.GetSessionDirectory();
    string path_obj = Path.Combine(sessionDir, "Control-obj.txt");

    string ctrlfileContent = File.ReadAllText(path_obj);

    var ctrl = (XNSE_Control)AppControl.Deserialize(ctrlfileContent).CloneAs();

    ctrl.InitialValues.Clear();
    ctrl.InitialValues_Evaluators.Clear();

    int RestartTimestepNumber = 66;
    ctrl.RestartInfo = new Tuple<Guid, BoSSS.Foundation.IO.TimestepNumber>(restartSession.ID, RestartTimestepNumber);
    ctrl.ReInitTimestepIndex = RestartTimestepNumber;
    ctrl.NoOfTimesteps = 100;

    ctrl.Option_LevelSetEvolution = LevelSetEvolution.StokesExtension;

    ctrl.NonLinearSolver.Globalization = Newton.GlobalizationOption.LineSearch;

    ctrl.ChangeBoundaryCondition("pressure_outlet_top", "Dong_OutFlow_top");
    ctrl.ChangeBoundaryCondition("pressure_outlet_front", "Dong_OutFlow_front");
    ctrl.ChangeBoundaryCondition("pressure_outlet_downstream", "Dong_OutFlow_downstream");

    //ctrl.DbPath = _DbPath;
    ctrl.savetodb = true;

    ctrl.SessionName = restartName;

    Controls.Add(ctrl);
}

In [ ]:
if (!restartRun) {
var C = new XNSE_Control();


C.SetDGdegree(k);
C.FieldOptions.Add(VariableNames.GravityZ, new FieldOpts() {
    SaveToDB = FieldOpts.SaveToDBOpt.TRUE
});


// physical parameters
C.PhysicalParameters.rho_A = densityDrop;
C.PhysicalParameters.mu_A = 96e-3;

C.PhysicalParameters.rho_B = density;
C.PhysicalParameters.mu_B = density * kinViscosity;

C.PhysicalParameters.Sigma = sigma;

C.PhysicalParameters.IncludeConvection = true;

    
if (restartRun) {
    int RestartTimestepNumber = 66;
    C.RestartInfo = new Tuple<Guid, BoSSS.Foundation.IO.TimestepNumber>(restartSession.ID, RestartTimestepNumber);
    C.ReInitTimestepIndex = RestartTimestepNumber;

    C.ChangeBoundaryCondition("pressure_outlet_top", "Dong_OutFlow_top");
    C.ChangeBoundaryCondition("pressure_outlet_front", "Dong_OutFlow_front");
    C.ChangeBoundaryCondition("pressure_outlet_downstream", "Dong_OutFlow_downstream");

} else {
    // set grid
    double l_radial = zTopStar; 
    double l_upstream = zTopStar / 2.0;
    double l_downstream = zTopStar / 2.0;
    double h_axial = zTopStar; 

    int res_radial = 1 * res_global; 
    int res_azimuthal = (2 * res_global) / 2;
    int res_axial = 1 * res_global;

    Grid3D grd = RotatingDiskSector_CartesianCutOut(radiusOP, l_radial, l_upstream, l_downstream, h_axial, res_radial, res_azimuthal, res_axial);
    C.SetGrid(grd);

    // initial conditions
    C.AddInitialValue("VelocityX#A", vonKarmanHAM_velX);
    C.AddInitialValue("VelocityY#A", vonKarmanHAM_velY);
    C.AddInitialValue("VelocityZ#A", vonKarmanHAM_velZ);
    // C.AddInitialValue("Pressure#B", vonKarman_P);

    // C.AddInitialValue("Phi", PhiFunc);
    // C.AddInitialValue("VelocityZ#A", InitVelocity);

}

C.AddInitialValue("GravityZ#A", GravityValue);
// C.AddInitialValue("GravityZ#B", GravityValue);

// boundary conditions
C.AddBoundaryValue("velocity_inlet_rotatingDisk", "VelocityX#A", RotatingDiskVelocityX);
C.AddBoundaryValue("velocity_inlet_rotatingDisk", "VelocityY#A", RotatingDiskVelocityY);

if (BC_tag2_top == 1) {
    C.AddBoundaryValue("velocity_inlet_top", "VelocityX#B", vonKarmanHAM_velX);
    C.AddBoundaryValue("velocity_inlet_top", "VelocityY#B", vonKarmanHAM_velY);
    C.AddBoundaryValue("velocity_inlet_top", "VelocityZ#B", vonKarmanHAM_velZ);
}

if (BC_tag4_front == 1) {
    C.AddBoundaryValue("velocity_inlet_front", "VelocityX#B", vonKarmanHAM_velX);
    C.AddBoundaryValue("velocity_inlet_front", "VelocityY#B", vonKarmanHAM_velY);
    C.AddBoundaryValue("velocity_inlet_front", "VelocityZ#B", vonKarmanHAM_velZ);
}

C.AddBoundaryValue("velocity_inlet_back", "VelocityX#A", vonKarmanHAM_velX);
C.AddBoundaryValue("velocity_inlet_back", "VelocityY#A", vonKarmanHAM_velY);
C.AddBoundaryValue("velocity_inlet_back", "VelocityZ#A", vonKarmanHAM_velZ);

C.AddBoundaryValue("velocity_inlet_upstream", "VelocityX#A", vonKarmanHAM_velX);
C.AddBoundaryValue("velocity_inlet_upstream", "VelocityY#A", vonKarmanHAM_velY);
C.AddBoundaryValue("velocity_inlet_upstream", "VelocityZ#A", vonKarmanHAM_velZ);

// C.AddBoundaryValue("velocity_inlet_downstream", "VelocityX#B", vonKarman_velX);
// C.AddBoundaryValue("velocity_inlet_downstream", "VelocityY#B", vonKarman_velY);
// C.AddBoundaryValue("velocity_inlet_downstream", "VelocityZ#B", vonKarman_velZ);
// C.AddBoundaryValue("pressure_dirichlet_downstream", "Pressure#B", vonKarman_P);


C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

C.ReInitPeriod = 4;

// C.SkipSolveAndEvaluateResidual = true;

// C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
C.NonLinearSolver.Globalization = Newton.GlobalizationOption.LineSearch;
// C.NonLinearSolver.ConvergenceCriterion = 1e-9;
C.NonLinearSolver.MaxSolverIterations = 50;

// C.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();

//C.AgglomerationThreshold = 0.2;

// C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
// C.Timestepper_LevelSetHandling = LevelSetHandling.None;
// C.Option_LevelSetEvolution = LevelSetEvolution.None;

C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
C.Option_LevelSetEvolution = LevelSetEvolution.StokesExtension;
C.TimeSteppingScheme = TimeSteppingScheme.ImplicitEuler;
C.dtFixed = 2.5e-5;
C.NoOfTimesteps = 4;
    
{
    C.AdaptiveMeshRefinement = false;
    C.activeAMRlevelIndicators.Add(new AMRwithGaussCheck() { maxRefinementLevel = AMRlevel_drop });
    //C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = AMRlevel_drop });
    //C.activeAMRlevelIndicators.Add(new AMRonNarrowbandAtBoundary(new byte[] { 1 }) { maxRefinementLevel = AMRlevel_dropBL });
    //C.activeAMRlevelIndicators.Add(new AMRLevelIndicatorLibrary.AMRonBoundary(new byte[] { 1 }) { maxRefinementLevel = AMRlevel_disk });
    C.AMR_startUpSweeps = AMRlevel_drop;
}

// C.PostprocessingModules.Add(new EnergyLogging());
//C.TracingNamespaces = "BoSSS.Solution.AdvancedSolvers,BoSSS.Foundation.ConstrainedDGprojection";
    
if (restartRun) 
    C.SessionName = restartName;
else {
    C.SessionName = $"DropletReboundGauthier_8x8x8AMR{AMRlevel_drop}_k{k}_NoDrop";
}
    
    
Controls.Add(C);
}

## Launch job

In [52]:
Controls.Select(C => C.SessionName)

[ DropletReboundGauthier_8x8x8AMR0_k3_ReI4_restart2_DongBC ]

In [53]:
myBatch.Name

FDY-WindowsHPC

In [54]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    int numThreads = 1;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

   
    if (myBatch.Name == "Lb2-specialPrj") {
        //oneJob.UseComputeNodesExclusive = true;
        ((SlurmClient)myBatch).ExecutionTime = "24:00:00";
    }

    oneJob.Activate(myBatch); 
}

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job DropletReboundGauthier_8x8x8AMR0_k3_ReI4_restart2_DongBC ... 


Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\DropletRebound_Gauthier-XNSE_Solver2025May08_112835.403602
copied 43 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.


## Wait for Completion and Check Job Status

In [ ]:
double hrsToWait = 0.25;

In [ ]:
wmg.BlockUntilAllJobsTerminate(hrsToWait*3600);

unable to determine job status - unknown
unable to determine job status - unknown
Timeout.


In [ ]:
var JobStat = Controls.Select(ctrl => ctrl.GetJob().Status).ToArray();
JobStat

index,value
0,InProgress


In [ ]:
NUnit.Framework.Assert.Zero(JobStat.Where(jS => (jS == BoSSS.Application.BoSSSpad.JobStatus.FailedOrCanceled)).Count(), "Some Jobs Failed");